#ASSIGNMENT 2 - DATA MINING
##HUDA SHAHAT 23I-5057

#import the required libraries

In [ ]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
from decimal import Decimal

#inspect the website first page

In [ ]:

url = "https://books.toscrape.com/"
response = requests.get(url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

books = []

for book in soup.select("article.product_pod"):
    title = book.select_one("h3 a")["title"]

    price_text = book.select_one("p.price_color").get_text(strip=True)
    price = float(
        price_text.replace("£", "").replace("Â", "")
    )

    rating_element = book.select_one("p.star-rating")
    rating_word = next(
        word for word in rating_element["class"]
        if word in rating_map
    )
    rating = rating_map[rating_word]

    availability = book.select_one(
        "p.instock.availability"
    ).get_text(" ", strip=True)

    books.append({
        "Title": title,
        "Price": price,
        "Rating": rating,
        "Availability": availability
    })

print("Number of books scraped:", len(books))
print(books[:3])


#scrape 5 pages or all catalogue pages

In [ ]:
import time

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

base_url = "https://books.toscrape.com/"

# BONUS 1: Keep this as 5 to scrape five pages.
# BOSS LEVEL: Change 5 to None to scrape all catalogue pages.
MAX_PAGES = 5

books = []
url = base_url
page_number = 0

while url and (MAX_PAGES is None or page_number < MAX_PAGES):
    page_number += 1
    print("Scraping page", page_number, "...")

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.select("article.product_pod"):
        title = book.select_one("h3 a")["title"]

        price_text = book.select_one("p.price_color").get_text(strip=True)
        price = float(price_text.replace("Â", "").replace("£", ""))

        rating_classes = book.select_one("p.star-rating")["class"]
        rating_word = next(word for word in rating_classes if word in rating_map)
        rating = rating_map[rating_word]

        availability = book.select_one(
            "p.instock.availability"
        ).get_text(" ", strip=True)

        # BONUS 2: Save the individual product page URL.
        product_url = urljoin(url, book.select_one("h3 a")["href"])

        books.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability,
            "URL": product_url
        })

    next_link = soup.select_one("li.next a")
    if next_link:
        url = urljoin(url, next_link["href"])
    else:
        url = None

    time.sleep(0.2)

df = pd.DataFrame(books)
df = df.drop_duplicates(subset="Title").reset_index(drop=True)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

print("\nTotal pages scraped:", page_number)
print("Total books scraped:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())


#clean and save dataset

In [ ]:

# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Remove duplicate records, if any
df = df.drop_duplicates(subset="Title").reset_index(drop=True)

# Make sure numeric columns have the correct data types
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

# Save the scraped dataset
df.to_csv("competitor_books.csv", index=False)

print("\nDataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nCSV saved successfully!")


#download csv

In [ ]:

from google.colab import files
files.download("competitor_books.csv")


#intelligence analysis

In [ ]:

# 1. Most expensive book
most_expensive = df.loc[df["Price"].idxmax()]

print("1. Most expensive book:")
print(most_expensive["Title"], "- £", round(most_expensive["Price"], 2))

# 2. Cheapest book
cheapest = df.loc[df["Price"].idxmin()]

print("\n2. Cheapest book:")
print(cheapest["Title"], "- £", round(cheapest["Price"], 2))

# 3. Average price
average_price = df["Price"].mean()

print("\n3. Average book price: £", round(average_price, 2))

# 4. Number of 5-star books
five_star_books = df[df["Rating"] == 5]

print("\n4. Number of 5-star books:", len(five_star_books))

# 5. Percentage of books in stock
in_stock = df["Availability"].str.contains(
    "in stock", case=False, na=False
)

stock_percentage = in_stock.mean() * 100

print("\n5. Percentage in stock:", round(stock_percentage, 2), "%")

# 6. Top 5 highest-rated books
top_five = df.sort_values(
    by=["Rating", "Price"],
    ascending=[False, True]
).head(5)

print("\n6. Top 5 highest-rated books:")
display(top_five[["Title", "Price", "Rating"]])

# 7. Cheapest book among 5-star books
if not five_star_books.empty:
    cheapest_five_star = five_star_books.loc[
        five_star_books["Price"].idxmin()
    ]

    print("\n7. Cheapest 5-star book:")
    print(
        cheapest_five_star["Title"],
        "- £", round(cheapest_five_star["Price"], 2)
    )
else:
    print("\n7. No 5-star books were found in this dataset.")


#Bonus Missions 3 and 4: Rating Counts and Bar Chart

In [ ]:
import matplotlib.pyplot as plt

# BONUS 3: Count books at each star-rating level (including zero counts).
rating_counts = (
    df["Rating"]
    .value_counts()
    .reindex([1, 2, 3, 4, 5], fill_value=0)
    .sort_index()
)

print("Number of books at each star rating:")
display(rating_counts.rename_axis("Star Rating").to_frame("Number of Books"))

# BONUS 4: Create a bar chart of rating counts.
rating_counts.plot(kind="bar")
plt.title("Number of Books by Star Rating")
plt.xlabel("Star Rating")
plt.ylabel("Number of Books")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


#Complete the £100 CEO Challenge
The goal is to select books with a combined price of no more than £100 while maximizing the sum of their star ratings. This is a small version of the knapsack optimization problem.

For example, a 5-star book contributes 5 points to your total rating score, while a 3-star book contributes 3 points. We want the best combination we can afford, not simply the most expensive books.

In [ ]:
import numpy as np

# CEO Challenge: choose books costing no more than £100
# while maximizing the sum of their star ratings.

budget = 10000  # £100 expressed in pennies

items = df.dropna(subset=["Price", "Rating"]).copy()
items = items[
    (items["Price"] > 0) &
    (items["Rating"].between(1, 5))
].reset_index(drop=True)

items["CostPennies"] = items["Price"].mul(100).round().astype(int)

# dp[b] = highest total rating score achievable with budget b or less.
# take[i, b] records whether book i was selected when processing budget b.
dp = np.zeros(budget + 1, dtype=int)
take = np.zeros((len(items), budget + 1), dtype=bool)

for i, item in items.iterrows():
    cost = int(item["CostPennies"])
    rating = int(item["Rating"])

    if cost > budget:
        continue

    # Go backwards so each book can be selected only once.
    for b in range(budget, cost - 1, -1):
        candidate_score = dp[b - cost] + rating
        if candidate_score > dp[b]:
            dp[b] = candidate_score
            take[i, b] = True

# Reconstruct the selected books.
remaining_budget = budget
selected_indices = []

for i in range(len(items) - 1, -1, -1):
    if take[i, remaining_budget]:
        selected_indices.append(i)
        remaining_budget -= int(items.loc[i, "CostPennies"])

selected_indices.reverse()
selected_books = items.iloc[selected_indices].copy()

print("Selected books:")
display(selected_books[["Title", "Price", "Rating"]])

total_cost = selected_books["Price"].sum()
total_rating = selected_books["Rating"].sum()

print(f"Total cost: £{total_cost:.2f}")
print("Total rating score:", int(total_rating))
print(f"Remaining budget: £{100 - total_cost:.2f}")


#Business Decision (recomendations)

In [ ]:
# Business Decision: evidence-based recommendation

# Select books that are highly rated, affordable, and in stock.
recommended = df[
    (df["Rating"] >= 4) &
    (df["Availability"].str.contains("in stock", case=False, na=False))
].sort_values(
    ["Rating", "Price"],
    ascending=[False, True]
)

print("Recommended books to prioritize:")
display(recommended[["Title", "Price", "Rating", "Availability", "URL"]].head(5))

# Calculate evidence from the current scraped dataset.
average_price = df["Price"].mean()
average_rating = df["Rating"].mean()
highly_rated_count = int((df["Rating"] >= 4).sum())
in_stock_count = int(
    df["Availability"].str.contains("in stock", case=False, na=False).sum()
)
in_stock_percentage = (
    in_stock_count / len(df) * 100 if len(df) else 0
)

print("\nEvidence from this dataset:")
print(f"Average book price: £{average_price:.2f}")
print(f"Average rating: {average_rating:.2f} out of 5")
print(f"Number of 4- and 5-star books: {highly_rated_count}")
print(f"Books in stock: {in_stock_count} out of {len(df)} ({in_stock_percentage:.2f}%)")

print("\nBusiness recommendation (based on the current results):")
print(
    f"I would prioritize highly rated, affordable books that are in stock. "
    f"In this dataset, {highly_rated_count} books have ratings of 4 or 5 stars, "
    f"and the average book price is £{average_price:.2f}. "
    f"I would compare the prices of these highly rated books and prioritize "
    f"the cheaper available options to offer customers good value."
)


## Business recommendation

After running the previous code cell, use the recommendation it prints in your assignment. The numbers are calculated from your current scraped dataset, so they update automatically if you scrape five pages or the full catalogue.

The recommendation should be 2–3 sentences and use at least two pieces of evidence, such as the number of 4- and 5-star books and the average book price. Do not leave old example numbers in your final submission.
